# Session 4 · Hypnogram & sleep statistics

So far we looked at the **EEG**. Today we look at the **hypnogram**: the experts' stage for every 30-s epoch, and the numbers clinicians and papers report from it (total sleep time, sleep efficiency, latencies, WASO, ...).

You will
1. turn the hypnogram into a YASA `Hypnogram` and plot it
2. compute a few **sleep statistics by hand**, then check them against YASA
3. see why sleep statistics use the **unedited** hypnogram (no artefact rejection)
4. look at **transitions** between stages, and the **transition rate** out of each stage
5. look at **bouts** (uninterrupted stretches of one stage) and the **sleep continuity index**
6. compare a **young and an old** subject.

**How to work**
- Mostly **run, change, look**: cells marked 👉 have a parameter to change and rerun. The `# TODO` cells are short.

In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import yasa

sys.path.insert(0, str(Path("..").resolve()))  # so Python finds sleep_utils.py in the repo folder
from sleep_utils import SUBJECTS_CSV, epoch_onsets, load_subject, make_epochs

mne.set_log_level("WARNING")

RESULTS_DIR = Path("..") / "results"
subjects = pd.read_csv(SUBJECTS_CSV)  # age, sex and lights-off of every night

## 1. The hypnogram as a YASA object

`make_epochs` gives the expert stage of every 30-s epoch, from lights-off to the end of the **sleep period**: the last sleep epoch before the first wake bout of 3 h or more. That's how `load_subject` crops the night, so naps the next day are left out (there is no lights-on time in Sleep-EDF). YASA wraps such a list of stages in a `yasa.Hypnogram`, which knows how to plot itself and compute statistics.

In [ ]:
raw = load_subject(0, 2)  # SC4002
epochs, stages = make_epochs(raw)
onsets = epoch_onsets(epochs)

hyp = yasa.Hypnogram(stages, freq="30s", start=onsets[0])
print(hyp)
hyp.hypno.head()  # a pandas Series: one stage per epoch, with its clock time

In [ ]:
hyp.plot_hypnogram(fill_color="gainsboro")
plt.title("SC4002, expert hypnogram")
plt.show()

**Questions**
1. How many sleep cycles do you count? Where in the night is most N3, and most REM?
2. `ART` (movement time) and `UNS` (unscored) are also "stages" here. Where are they?

## 2. Sleep statistics by hand

The basic definitions (all in minutes, one epoch = 0.5 min):

| | Definition |
|---|---|
| **TIB** time in bed | all epochs of the hypnogram (here: lights-off → end of the sleep period) |
| **TST** total sleep time | epochs in N1, N2, N3 or REM |
| **SE** sleep efficiency | TST / TIB × 100 |
| **SOL** sleep onset latency | from the start to the first sleep epoch |
| **WASO** wake after sleep onset | WAKE epochs after the first sleep epoch |

`hyp.hypno` is a pandas Series of stages, so `hyp.hypno.isin([...])` gives True/False per epoch, and `.sum()` counts the True.

### Exercise 2.1

In [ ]:
EPOCH_MIN = 30 / 60  # epoch duration in minutes: 30 seconds = 0.5 minutes
SLEEP = ["N1", "N2", "N3", "REM"]
h = hyp.hypno.to_numpy()  # the stages as a numpy array

is_sleep = np.isin(h, SLEEP)
first_sleep = np.flatnonzero(is_sleep)[0]  # index of the first sleep epoch

TIB = len(h) * EPOCH_MIN
# TODO: TST in minutes. Hint: is_sleep.sum() counts the sleep epochs
TST = ...
# TODO: SE in %
SE = ...
SOL = first_sleep * EPOCH_MIN
# TODO: WASO in minutes: WAKE epochs from first_sleep on. Hint: h[first_sleep:] == "WAKE"
WASO = ...

mine = pd.Series({"TIB": TIB, "TST": TST, "SE": SE, "SOL": SOL, "WASO": WASO})
mine.round(1)

### Now YASA

`hyp.sleep_statistics()` returns all the usual statistics at once. The definitions are in the documentation: https://yasa-sleep.org/generated/yasa.Hypnogram.html (method `sleep_statistics`).

In [ ]:
stats = pd.Series(hyp.sleep_statistics())
pd.DataFrame({"yours": mine, "YASA": stats[mine.index]}).round(1)

All of YASA's statistics (stages in minutes, `%N1`… in % of TST):

In [ ]:
stats.round(1)

**Questions**
1. Do your numbers match YASA's? If not, which definition differs?
2. YASA reports a few more: **SPT** (sleep period time), **SME** (sleep maintenance efficiency), **SFI** (sleep fragmentation index), **Lat_REM**. Look up what they mean.

### Exercise 2.2: REM latency

The AASM defines REM latency as the time from **sleep onset** to the first REM epoch.

In [ ]:
first_rem = np.flatnonzero(h == "REM")[0]
# TODO: REM latency in minutes, from the first sleep epoch to the first REM epoch
lat_rem = ...
print(f"REM latency, from sleep onset: {lat_rem} min")
print(f"YASA Lat_REM: {stats['Lat_REM']} min")

**Question:** From where does YASA count `Lat_REM`? Which one would you report in a paper, and what must you write in the methods?

## 3. Why sleep statistics use the unedited hypnogram

This morning you saved the bad epochs of SC4002 in `results/SC4002/bad_epochs_auto.csv`. What if we dropped them before computing the sleep statistics?

(No file? The cell recomputes it with `bad_epochs_table` from `sleep_utils`, which is this morning's detector.)

In [ ]:
bad_file = RESULTS_DIR / "SC4002" / "bad_epochs_auto.csv"
if bad_file.exists():
    bad = pd.read_csv(bad_file)["bad"].to_numpy()
else:
    from sleep_utils import bad_epochs_table
    epochs_filt, _ = make_epochs(raw.copy().filter(0.1, 40, picks=["eeg", "eog"]))
    bad = bad_epochs_table(epochs_filt, stages)["bad"].to_numpy()
print(f"{bad.sum()} bad epochs")

hyp_dropped = yasa.Hypnogram(h[~bad], freq="30s")  # the bad epochs removed
cols = ["TIB", "TST", "SE", "SOL", "WASO", "WAKE", "%N3", "%REM"]
pd.DataFrame({"unedited": stats[cols], "bad epochs dropped": pd.Series(hyp_dropped.sleep_statistics())[cols]}).round(1)

**Questions**
1. Which statistics change most? Why WASO?
2. A movement or muscle burst during sleep is often an **arousal or an awakening**: part of the sleep you want to measure. What would dropping those epochs do to a patient with fragmented sleep?
3. So the rule for the rest of the course: sleep statistics use the **unedited** expert hypnogram; artefact rejection only removes epochs from **EEG** features (spectra, band power, spindles, slow waves).

## 4. Transitions between stages

`hyp.transition_matrix()` counts how often each stage is followed by each other stage (epoch to epoch). It returns the **counts** and the **probabilities** (each row sums to 1).

In [ ]:
counts, probs = hyp.transition_matrix()
order = ["WAKE", "N1", "N2", "N3", "REM"]
probs = probs.reindex(index=order, columns=order)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(probs, cmap="Blues", vmin=0, vmax=1)
for i in range(len(order)):
    for j in range(len(order)):
        ax.text(j, i, f"{probs.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if probs.iloc[i, j] > 0.5 else "black", fontsize=9)
ax.set(xticks=range(5), xticklabels=order, yticks=range(5), yticklabels=order, xlabel="To", ylabel="From",
       title="SC4002: P(next stage | stage)")
plt.show()
counts.reindex(index=order, columns=order)

### Exercise 4.1: stage shifts and awakenings

A **stage shift** is any change of stage from one epoch to the next. An **awakening** is a sleep epoch followed by a WAKE epoch.

In [ ]:
sleep_wake = np.where(np.isin(h, SLEEP), "sleep", h)  # all sleep stages -> "sleep"
# TODO: the number of stage shifts: epochs whose stage differs from the next one. Hint: h[1:] != h[:-1]
n_shifts = ...
# TODO: the number of awakenings: a sleep epoch followed by a WAKE epoch
n_awakenings = ...
print(f"{n_shifts} stage shifts, {n_awakenings} awakenings, {n_shifts / (TST / 60):.1f} shifts per hour of sleep")
print(f"Awakenings per hour of sleep: {n_awakenings / (TST / 60):.2f} (YASA's SFI: {stats['SFI']:.2f})")

### Exercise 4.2: transition rate from a stage

How quickly do we leave a stage once we are in it? The **transition rate from a stage** is the number of times the stage is left (the epoch is in that stage, the next one is not), divided by the minutes spent in that stage. Unit: transitions per minute.

In [ ]:
def transition_rate_from_stage(h, stage):
    """Transitions out of `stage` per minute spent in `stage`."""
    # TODO: the number of epochs in `stage` followed by an epoch NOT in `stage`
    n_exits = ...
    # TODO: the minutes spent in `stage`
    minutes = ...
    return n_exits / minutes if minutes > 0 else np.nan


rates = pd.Series({stage: transition_rate_from_stage(h, stage) for stage in order})
rates.round(3)

**Questions**
1. Which stage is the most stable? Which one is mostly a "transition" stage? Do the transition rates tell the same story as the matrix?
2. From which stage do awakenings come most often? Look at the `WAKE` column of the counts.
3. N3 → REM is (almost) never seen. Why?

## 5. Bouts and the sleep continuity index

A **bout** is an uninterrupted stretch of the same stage: `N2 N2 N2 N3 N3 N2` has three bouts (N2 of 1.5 min, N3 of 1 min, N2 of 0.5 min). Fragmented sleep has many short bouts.

`find_bouts` (given) returns one row per bout: its stage, its first epoch and its length in epochs.

In [ ]:
def find_bouts(h):
    """One row per bout (consecutive epochs of the same stage): stage, first epoch, number of epochs."""
    new_bout = np.r_[True, h[1:] != h[:-1]]  # True where a new bout starts
    bout_id = np.cumsum(new_bout)            # 1, 1, 1, 2, 2, 3, ...
    table = pd.DataFrame({"stage": h, "bout": bout_id, "epoch": np.arange(len(h))})
    return table.groupby("bout").agg(stage=("stage", "first"), start=("epoch", "first"),
                                     n_epochs=("epoch", "size")).reset_index(drop=True)


bouts = find_bouts(h)
print(len(bouts), "bouts")
bouts.head(10)

### Exercise 5.1: bout durations

In [ ]:
# TODO: the duration of each bout in minutes
bouts["duration_min"] = ...
bouts["start_h"] = bouts["start"] * EPOCH_MIN / 60  # hours after lights-off

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].boxplot([bouts.loc[bouts["stage"] == s, "duration_min"] for s in order], tick_labels=order)
axes[0].set(ylabel="Bout duration (min)", title="Bout durations per stage")
for s, color in [("N3", "tab:blue"), ("REM", "tab:red")]:
    d = bouts[bouts["stage"] == s]
    axes[1].scatter(d["start_h"], d["duration_min"], color=color, label=s)
axes[1].set(xlabel="Time after lights-off (h)", ylabel="Bout duration (min)", title="N3 and REM bouts across the night")
axes[1].legend()
plt.show()

# TODO: per stage, the number, median, mean and max of the bout durations
# Hint: bouts.groupby("stage")["duration_min"].agg([...])
...

**Questions**
1. How long is the longest REM bout, and the longest N3 bout? What does a "typical" N1 bout look like?
2. Compare the mean bout duration of each stage with 1 / transition rate (Exercise 4.2). What do you notice?
3. Do N3 and REM bouts change across the night?

### Exercise 5.2: sleep continuity index

The **sleep continuity index** (SCI) is the mean duration of the **sleep bouts** (bouts of N1, N2, N3 or REM), in minutes. High = long uninterrupted stretches of sleep, low = fragmented sleep.

In [ ]:
def sleep_continuity_index(h):
    """Mean duration (min) of the bouts of a sleep stage (N1, N2, N3, REM)."""
    b = find_bouts(h)
    # TODO: keep only the bouts of a sleep stage, and return their mean duration in minutes
    sleep_bouts = ...
    return ...


print(f"SCI: {sleep_continuity_index(h):.2f} min")

**Question:** A change from N2 to N3 starts a new bout here, although the subject stays asleep. Is that what you want for a *continuity* index? What would the alternative be?

## 6. A young and an old subject

👉 Pick subjects in `SC-subjects.csv` (e.g. `subjects.sort_values("age").tail(10)`), change `NIGHTS` and rerun.

In [ ]:
NIGHTS = [(0, 2), (..., ...)]  # 👉 (subject, night): SC4002 is 33 years old, SC4752 is 96 (the night from the T2 slide)
cols = ["TIB", "TST", "SE", "WASO", "SFI", "%N3", "%REM"]

fig, axes = plt.subplots(len(NIGHTS), 1, figsize=(10, 2.8 * len(NIGHTS)), squeeze=False)
table = {}
for ax, (subject, night) in zip(axes[:, 0], NIGHTS):
    age = subjects.loc[(subjects["subject"] == subject) & (subjects["night"] == night), "age"].iloc[0]
    ep, st = make_epochs(load_subject(subject, night))
    hy = yasa.Hypnogram(st, freq="30s", start=epoch_onsets(ep)[0])
    hy.plot_hypnogram(ax=ax, fill_color="gainsboro")
    ax.set_title(f"SC4{subject:02d}{night}, {age} years")

    row = pd.Series(hy.sleep_statistics())[cols]
    row["SCI"] = sleep_continuity_index(st)
    for stage in ["N1", "N2", "N3", "REM"]:
        row[f"rate_{stage}"] = transition_rate_from_stage(st, stage)
    table[f"SC4{subject:02d}{night} ({age} y)"] = row
plt.tight_layout()
plt.show()
pd.DataFrame(table).round(2)

**Questions**
1. What differs between the two nights? Is that what you expected with age (T2)?
2. One night per person: what else could explain the difference? (Tonight we run this on the **whole cohort**.)

## Optional exercises

1. **Cycles.** Split the night into NREM–REM cycles (a simple rule: a cycle ends at the end of a REM period that lasts at least 5 min). How many cycles, how long, and how does %N3 change from cycle to cycle?
2. **Smoothing the hypnogram?** Some pipelines merge very short bouts (e.g. a single N1 epoch inside N2). Write a function that does it and see how the SCI, the transition rates and the number of stage shifts change. Why is this risky?
3. **Re-entry latency.** After leaving REM, how long does it take until the next REM bout starts? Compute the mean REM re-entry latency for the young and the old night.

In [ ]:
# Optional exercise 1

In [ ]:
# Optional exercise 2

In [ ]:
# Optional exercise 3